# Adaptive Learning and Trajectory Correction Analysis

This notebook analyzes the behavior of the adaptive learning mechanisms, tracks parameter drift over time, and evaluates the effectiveness of the trajectory correction logic.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta, timezone

from src.analysis.data_loader import DataLoader

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_context("notebook")
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Data Loading

Load data related to adaptive learning parameters and trajectory corrections.

In [ ]:
loader = DataLoader()
end_time = datetime.now(timezone.utc)
start_time = end_time - timedelta(days=14)

features = [
    'indoor_temperature',
    'ml_target_temperature',
    'trajectory_correction_applied',
    'learned_heat_capacity',
    'learned_thermal_resistance',
    'prediction_error'
]

try:
    df = loader.fetch_training_data(start_time=start_time, end_time=end_time, features=features)
    print(f"Loaded {len(df)} records from {start_time.strftime('%Y-%m-%d')} to {end_time.strftime('%Y-%m-%d')}")
except Exception as e:
    print(f"Error loading data: {e}")
    # Create dummy data for demonstration
    dates = pd.date_range(start=start_time, end=end_time, freq='1h')
    df = pd.DataFrame(index=dates)
    df['indoor_temperature'] = 20 + np.sin(np.linspace(0, 20, len(dates))) + np.random.normal(0, 0.2, len(dates))
    df['ml_target_temperature'] = 21
    df['trajectory_correction_applied'] = np.where(np.abs(df['indoor_temperature'] - df['ml_target_temperature']) > 1.5, 1, 0)
    df['learned_heat_capacity'] = 5000 + np.cumsum(np.random.normal(0, 10, len(dates)))
    df['learned_thermal_resistance'] = 0.05 + np.cumsum(np.random.normal(0, 0.0001, len(dates)))
    df['prediction_error'] = np.random.normal(0, 0.5, len(dates))
    print("Using generated dummy data for demonstration.")

## 2. Parameter Drift Analysis

Analyze how learned parameters (e.g., heat capacity, thermal resistance) evolve over time.

In [ ]:
if 'learned_heat_capacity' in df.columns and 'learned_thermal_resistance' in df.columns:
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), sharex=True)
    
    ax1.plot(df.index, df['learned_heat_capacity'], color='purple')
    ax1.set_title('Learned Heat Capacity Over Time')
    ax1.set_ylabel('Heat Capacity (J/K)')
    
    ax2.plot(df.index, df['learned_thermal_resistance'], color='brown')
    ax2.set_title('Learned Thermal Resistance Over Time')
    ax2.set_ylabel('Thermal Resistance (K/W)')
    ax2.set_xlabel('Time')
    
    plt.tight_layout()
    plt.show()
else:
    print("Learned parameter data not available.")

## 3. Trajectory Correction Effectiveness

Evaluate when trajectory corrections are applied and their impact on reducing prediction error.

In [ ]:
if 'trajectory_correction_applied' in df.columns and 'prediction_error' in df.columns:
    corrections = df[df['trajectory_correction_applied'] > 0]
    print(f"Total trajectory corrections applied: {len(corrections)}")
    
    fig, ax1 = plt.subplots(figsize=(14, 6))
    
    color = 'tab:blue'
    ax1.set_xlabel('Time')
    ax1.set_ylabel('Prediction Error (°C)', color=color)
    ax1.plot(df.index, df['prediction_error'], color=color, alpha=0.6, label='Prediction Error')
    ax1.tick_params(axis='y', labelcolor=color)
    
    # Highlight corrections
    ax1.scatter(corrections.index, corrections['prediction_error'], color='red', zorder=5, label='Correction Applied')
    
    ax1.axhline(0, color='black', linestyle='--', alpha=0.5)
    ax1.legend(loc='upper right')
    
    plt.title('Prediction Error and Trajectory Corrections')
    plt.show()
    
    # Analyze error distribution before and after corrections (simplified)
    plt.figure(figsize=(10, 5))
    sns.histplot(df['prediction_error'].dropna(), bins=40, kde=True)
    plt.title('Distribution of Prediction Errors')
    plt.xlabel('Error (°C)')
    plt.show()
else:
    print("Trajectory correction or prediction error data not available.")